In [1]:
from evidently import DataDefinition, Report, Dataset
from evidently.metrics import QuantileValue, MissingValueCount, UniqueValueCount, AlmostConstantColumnsCount
import pandas as pd
import warnings


warnings.filterwarnings("ignore", category=RuntimeWarning)

## Q1. Prepare the dataset

In [2]:
file_name = "data/green_tripdata_2024-03.parquet"

In [3]:
def read_file(file_name):
    df = pd.read_parquet(file_name)
    df = df[(df['lpep_pickup_datetime'] >= '2024-03-01') & (df['lpep_pickup_datetime'] < '2024-04-01')]
    return df
df = read_file(file_name)
print(f"number of rows: {len(df)}")

number of rows: 57447


In [4]:
categorical_cols = ['RatecodeID', 'passenger_count', 'payment_type', 'trip_type', 'congestion_surcharge', 'store_and_fwd_flag']

df[categorical_cols] = df[categorical_cols].astype("str")
eval_data = Dataset.from_pandas(df, data_definition=DataDefinition())

In [5]:
report = Report(
    metrics=[
        QuantileValue(column="fare_amount", quantile=0.5),
        MissingValueCount(column="trip_distance"),
        # UniqueValueCount(column="trip_distance"),
        AlmostConstantColumnsCount(column="trip_distance", threshold=0.95)
    ]
)

## Simulate Real world Production Environment of Reciving data every day

In [6]:
df['date'] = df['lpep_pickup_datetime'].dt.date
daily_groups = df.groupby('date')

In [11]:
max_median = 0
for date, daily_df in daily_groups:
    # report.run(current_data=Dataset.from_pandas(group, data_definition=DataDefinition()))
    rep = report.run(current_data=daily_df, reference_data=None)
    result_dict = rep.dict()
    median = result_dict["metrics"][0]["value"]
    print(f"result dict: {result_dict["metrics"]}")
    if median > max_median:
        max_median = median

print(f"Max median fare amount across all dates: {max_median}")

result dict: [{'id': '641f3d487377ef8aeba674733f8705f4', 'metric_name': 'QuantileValue(column=fare_amount,quantile=0.5)', 'config': {'type': 'evidently:metric_v2:QuantileValue', 'column': 'fare_amount', 'quantile': 0.5}, 'value': np.float64(13.5)}, {'id': '863d86acbb8f0bb20f51e846873181fb', 'metric_name': 'MissingValueCount(column=trip_distance)', 'config': {'type': 'evidently:metric_v2:MissingValueCount', 'column': 'trip_distance'}, 'value': {'count': 0.0, 'share': np.float64(0.0)}}, {'id': '9e9fa5ec57528fadbf97a35ad5b66838', 'metric_name': 'AlmostConstantColumnsCount()', 'config': {'type': 'evidently:metric_v2:AlmostConstantColumnsCount'}, 'value': 4.0}]
result dict: [{'id': '641f3d487377ef8aeba674733f8705f4', 'metric_name': 'QuantileValue(column=fare_amount,quantile=0.5)', 'config': {'type': 'evidently:metric_v2:QuantileValue', 'column': 'fare_amount', 'quantile': 0.5}, 'value': np.float64(13.5)}, {'id': '863d86acbb8f0bb20f51e846873181fb', 'metric_name': 'MissingValueCount(column=tr

In [8]:
# report.run(current_data=eval_data)